In [ ]:
# === Imports ===
# Standard library
import ctypes
import gc
import json
import os

# Third-party
import numpy as np
import orjson
import pandas as pd
import psutil
from tqdm import tqdm

In [ ]:
# FOR G-DRIVE USE
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# NOTE: Only needed for the Colab / Google-Drive version.
# For local runs, install dependencies from requirements.txt instead.
# !pip install orjson

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.0/134.0 kB 7.4 MB/s eta 0:00:00


In [ ]:
REALLY_BIG_FILE = False

def force_free_memory():
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)

def mem_report(label):
    mem = psutil.virtual_memory()
    print(f"[MEM {label}] Used: {mem.used / (1024**3):.1f} GB / {mem.total / (1024**3):.1f} GB | Available: {mem.available / (1024**3):.1f} GB")

# g_drive_path = "drive/MyDrive/AMR/embeddings"
path = "intermediate_outputs/embeddings"
for file in tqdm(os.listdir(path), desc="Embedding Files"):
    if os.path.exists(f"{path}/hypernym_substitution/{file}"):
        print(f"Skipping (already processed): {file}")
        continue
    else:
        print(f"Loading file: {file}")

    # g_drive_path = f"drive/MyDrive/AMR/embeddings/{file}"
    path = f"{path}/{file}"
    mem_report("before json.load")
    with open(path, "rb") as f:  # orjson needs bytes
        d = orjson.loads(f.read())
    mem_report("after json.load")
    metadata = d["metadata"]
    data = d["data"]
    eng_code = "en_EN" if "wmt" in file else "eng_Latn"

    for aug_category in tqdm(metadata["foil_types"], desc="Augmentation types", leave=False):
        #g_drive_out path = f"drive/MyDrive/AMR/similarities/{aug_category}/{file}"
        g_drive_out_path = f"intermediate_outputs/similarities/{aug_category}/{file}"
        print(f"Calculating: {out_path}")
        rows = []
        for lang in metadata["languages"]:
            for idx in data.keys():
                if aug_category not in data[idx].keys():
                    continue
                rows.append({
                    "idx": idx,
                    "filename": path.split("/")[-1],
                    "model": "_".join(path.split("/")[-1].replace(".json", "").split("_")[1:]),
                    "dataset": path.split("/")[-1].replace(".json", "").split("_")[0],
                    "level": data[idx].get("level"),
                    "aug_type": aug_category,
                    "lang_pair": f"{eng_code}-{lang}",
                    "text_en": data[idx][eng_code]["original_text"],
                    "text_en_neg": data[idx][aug_category]["text"],
                    "text_other": data[idx][lang]["original_text"],
                    "sim_parallel": np.dot(
                        data[idx][eng_code]["embedding"],
                        data[idx][lang]["embedding"]
                    ),
                    "sim_neg": np.dot(
                        data[idx][eng_code]["embedding"],
                        data[idx][aug_category]["embedding"]
                    ),
                    "sim_x_neg": np.dot(
                        data[idx][aug_category]["embedding"],
                        data[idx][lang]["embedding"]
                    ),
                })

        print(f"Total rows: {len(rows)}")
        mem_report("after building rows")

        if REALLY_BIG_FILE:
            del d, data
            force_free_memory()
            mem_report("after deleting d/data")

        df = pd.DataFrame(rows)
        mem_report("after creating DataFrame")
        del rows
        force_free_memory()
        mem_report("after deleting rows")

        os.makedirs(f"intermediate_outputs/similarities/{aug_category}/", exist_ok=True)
        df.to_json(out_path, orient="records", lines=True, force_ascii=False)
        mem_report("after to_json")
        print(f"Saved: {out_path}")
        del df
        force_free_memory()
        mem_report("after cleanup")

        if REALLY_BIG_FILE:
            with open(path, "rb") as f:
                d = orjson.loads(f.read())
            metadata = d["metadata"]
            data = d["data"]
            mem_report("after reload")

Embedding Files:   2%|▏         | 1/51 [00:00<00:16,  2.96it/s]

Skipping (already processed): wmt24pp_Alibaba_NLP_gte_multilingual_base.json
Skipping (already processed): flores200_Alibaba_NLP_gte_multilingual_base.json
Skipping (already processed): wmt24pp_BAAI_bge_m3.json
Skipping (already processed): wmt24pp_sentence_transformers_LaBSE.json
Skipping (already processed): wmt24pp_google_embeddinggemma_300m.json
Skipping (already processed): wmt24pp_nomic_embed_text_v2_moe.json
Skipping (already processed): wmt24pp_multilingual_e5_large.json
Skipping (already processed): wmt24pp_jina_embeddings_v5_text_small.json
Skipping (already processed): wmt24pp_jina_embeddings_v5_text_nano.json
Skipping (already processed): wmt24pp_jina_embeddings_v3.json
Skipping (already processed): wmt24pp_multilingual_e5_large_instruct.json
Skipping (already processed): wmt24pp_multilingual_e5_base.json
Skipping (already processed): wmt24pp_paraphrase_multiL_mpnet_base.json
Skipping (already processed): flores200_paraphrase_multiL_mpnet_base.json
Skipping (already process


Augmentation types:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating: drive/MyDrive/AMR/similarities/polarity_negation/bouquet_Qwen3_Embedding_0.6B.json
Total rows: 183975
[MEM after building rows] Used: 11.4 GB / 172.9 GB | Available: 160.0 GB
[MEM after creating DataFrame] Used: 11.4 GB / 172.9 GB | Available: 160.0 GB
[MEM after deleting rows] Used: 11.1 GB / 172.9 GB | Available: 160.3 GB
[MEM after to_json] Used: 11.3 GB / 172.9 GB | Available: 160.1 GB
Saved: drive/MyDrive/AMR/similarities/polarity_negation/bouquet_Qwen3_Embedding_0.6B.json



Augmentation types:  25%|██▌       | 1/4 [00:57<02:51, 57.27s/it]

...

[MEM after cleanup] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
Calculating: drive/MyDrive/AMR/similarities/antonym_replacement/bouquet_Qwen3_Embedding_8B.json
Total rows: 164175
[MEM after building rows] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
[MEM after creating DataFrame] Used: 39.3 GB / 172.9 GB | Available: 132.0 GB
[MEM after deleting rows] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
[MEM after to_json] Used: 41.7 GB / 172.9 GB | Available: 129.7 GB
Saved: drive/MyDrive/AMR/similarities/antonym_replacement/bouquet_Qwen3_Embedding_8B.json



Augmentation types:  75%|███████▌  | 3/4 [08:14<02:42, 162.75s/it]

[MEM after cleanup] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
Calculating: drive/MyDrive/AMR/similarities/hypernym_substitution/bouquet_Qwen3_Embedding_8B.json
Total rows: 93775
[MEM after building rows] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
[MEM after creating DataFrame] Used: 39.3 GB / 172.9 GB | Available: 132.0 GB
[MEM after deleting rows] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB
[MEM after to_json] Used: 41.1 GB / 172.9 GB | Available: 130.3 GB
Saved: drive/MyDrive/AMR/similarities/hypernym_substitution/bouquet_Qwen3_Embedding_8B.json



Embedding Files: 100%|██████████| 51/51 [1:18:35<00:00, 92.47s/it] 

[MEM after cleanup] Used: 39.3 GB / 172.9 GB | Available: 132.1 GB


In [ ]:
# gold_len = len(os.listdir("drive/MyDrive/AMR/embeddings/"))
gold_len = len(os.listdir("intermediate_outputs/embeddings/"))
#
for aug in metadata["foil_types"]:
    # g_drive_path = f"drive/MyDrive/AMR/similarities/{aug}"
    path = f"intermediate_outputs/similarities/{aug}"
    files = os.listdir(path)
    if len(files) != gold_len:
        print(f"❌ {path}: {len(files)} files")
        print("   Files:", files)